In [3]:
import os
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [4]:
from openai import OpenAI

client = OpenAI(
    base_url=os.environ["OPENAI_BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

MODEL = os.environ["MODEL_NAME"]



In [5]:
def ask(messages, **kwargs):
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        **kwargs,
    )
    return response.choices[0].message.content

In [6]:
prompt = "Classify the sentiment as positive, negative, or neutral: 'The food was cold and the service was slow.'"
print(ask([{"role": "user", "content": prompt}]))

Negative


In [7]:
prompt = """Classify the sentiment.

Text: I love this phone!
Sentiment: positive

Text: It broke after two days.
Sentiment: negative

Text: The package arrived on Tuesday.
Sentiment: neutral

Text: The battery life is amazing.
Sentiment:"""
print(ask([{"role": "user", "content": prompt}]))

positive


In [8]:
messages = [
    {"role": "system", "content": "You are a strict Python code reviewer. Be concise."},
    {"role": "user", "content": "Review: def add(a,b): return a+b"},
]
print(ask(messages))

**Issues:**

1. **No type hints** — unclear what types are supported; `add("a","b")` silently concatenates, `add([1],[2])` silently concatenates lists. Ambiguous contract.
2. **No docstring** — trivial here, but establishes the pattern.
3. **No input validation** — no handling of `None` or mismatched types; failure mode is whatever Python's `+` does, which may be silent (str/list concat) rather than an error.
4. **Missing spacing** — `def add(a, b):` and `return a + b` per PEP 8.
5. **Shadows nothing critical, but `add` is vague** — fine as a helper, not as a public API name.
6. **No tests** — even trivial functions should have one or two.

**Suggested:**

```python
def add(a: float, b: float) -> float:
    """Return the sum of two numbers."""
    return a + b
```

If numeric-only is the intent, note that `+` on non-numeric types won't raise — consider `isinstance` checks or `numbers.Number` if callers may pass unexpected types.

Nothing here is *wrong* for a throwaway snippet; it's ju

In [11]:
prompt = """A shop sells pens at 12 rupees each. Ravi buys 7 pens and pays with a 100 rupee note.
How much change does he get? Think step by step, then give the final answer."""
print(ask([{"role": "user", "content": prompt}]))

7 pens cost \(7 \times 12 = 84\) rupees.

Change from 100 rupees:

\[
100 - 84 = 16
\]

**Ravi gets 16 rupees change.**


In [12]:
prompt = """A shop sells pens at 12 rupees each. Ravi buys 7 pens and pays with a 100 rupee note.
How much change does he get?"""
print(ask([{"role": "user", "content": prompt}]))

Ravi spends \(7 \times 12 = 84\) rupees.

Change from ₹100:  
\(100 - 84 = 16\)

**He gets ₹16 change.**


def ask(messages, **kwargs):
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        **kwargs,
    )
    return response.choices[0].message.content

In [13]:
import json

prompt = """Extract the details from this text and return only valid JSON with keys name, age, city.

Text: Priya is 28 years old and lives in Chennai."""
result = ask(
    [{"role": "user", "content": prompt}],
    response_format={"type": "json_object"},
)
print(json.loads(result))

{'name': 'Priya', 'age': 28, 'city': 'Chennai'}


In [15]:
article = "Transformers use self-attention to process sequences in parallel, replacing recurrence..."

summary = ask([{"role": "user", "content": f"Summarize in 2 sentences:\n{article}"}])
tweet = ask([{"role": "user", "content": f"Turn this summary into a tweet:\n{summary}"}])
print(summary)
print("\n")
print(tweet)

Transformers process sequences in parallel using self-attention rather than recurrence. This allows them to model sequential data more efficiently.
Transformers process sequences in parallel using self-attention instead of recurrence—making them more efficient at modeling sequential data. ⚡ #AI #MachineLearning


In [18]:
from collections import Counter

question = "If a train travels 100 km in 45 minutes, what is its speed in km/h? Reply with just the number."
answers = [
    ask([{"role": "user", "content": question}], temperature=1.5).strip()
    for _ in range(5)
]
print(answers)
answers = ['80', '80', '90', '90', '90']
print("Majority answer:", Counter(answers).most_common(1)[0][0])

['133.33', '133.33', '133.33', '133.33', '133.33']
Majority answer: 90


In [ ]:
from collections import Counter

question = "If a train travels 60 km in 45 minutes, what is its speed in km/h? Reply with just the number."

answers = []   
for _ in range(5):
        answers.append(ask([{"role": "user", "content": question}], temperature=0.8).strip())

print(answers)
print("Majority answer:", Counter(answers).most_common(1)[0][0])

In [21]:
history = []

def chat(user_input):
    print("\n ===== ")
    print(history)

    history.append({"role": "user", "content": user_input})
    reply = ask(history)
    history.append({"role": "assistant", "content": reply})
    return reply

#print(chat("My name is Arun."))

print(chat("What is my name?"))


 ===== 
[]
I don’t know your name—you haven’t provided it in this conversation. If you tell me, I’ll use it.


In [ ]:
# LLM Model -> it will generate one token at time/predicting next token

# LLM Model -> it will only knowledge or information current req./call